# Practical 00: CSV Inspection and Cleaning (00try2)

This notebook is the notebook version of 00try2.py with theory plus runnable code.

## Common Workflow
1. Import libraries  
2. Load CSV (pandas)  
3. Inspect (head, info, isnull)  
4. Clean (nulls, duplicates, text)  
5. Apply method (CSV validation checks)  
6. Visualize  
7. Interpret

## Step 1: Import Libraries
Theory: pandas is used for CSV loading, cleaning, type handling, and validation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Step 2 and 3: Load and Inspect
Theory: Always begin with df.info() and df.isnull().sum() to understand schema and missing values.

In [ ]:
df = pd.read_csv('new_ev_tech_cleaned_dataset.csv', encoding='ISO-8859-1')
print(df.head())

print('\nData Description:\n', df.describe(include='all'))
print('\nData Info:')
df.info()
print('\nNull Values:\n', df.isnull().sum())
print('\nShape:', df.shape)

## Step 4: Clean Dataset
Theory: Standardize text to improve consistency for downstream NLP and analytics tasks.

In [ ]:
df = df.drop_duplicates().copy()

df['cleaned_text'] = df['Text'].fillna('').astype(str).str.lower().str.strip()
df['cleaned_text'] = df['cleaned_text'].str.replace(r'[^\w\s]', '', regex=True)
df['cleaned_text'] = df['cleaned_text'].str.replace(r'[^a-z\s]', '', regex=True)
print('Cleaned Text sample:\n', df['cleaned_text'].head())

print('\nData types before casting:\n', df.dtypes)
df['Likes'] = pd.to_numeric(df['Likes'], errors='coerce').fillna(0).astype(int)
df['View_Count'] = pd.to_numeric(df['View_Count'], errors='coerce').fillna(0).astype(int)
df['Timestamp'] = pd.to_datetime(df['Timestamp'], utc=True, errors='coerce')
print('\nAfter casting:')
print(df[['Likes', 'View_Count', 'Timestamp']].dtypes)

## Step 5: Apply CSV Validation Checks
Theory: Validate null handling, quoting behavior, long text behavior, and malformed-row safety.

In [ ]:
print('Nulls before handling:\n', df.isnull().sum())

df['Parent_ID'] = df['Parent_ID'].fillna('ROOT')
df.dropna(subset=['Text'], inplace=True)
print('\nParent_ID nulls after fill:', df['Parent_ID'].isnull().sum())
print('Rows after dropna on Text:', len(df))

has_comma = df['Text'].str.contains(',', na=False).sum()
has_quote = df['Text'].str.contains(chr(34), na=False).sum()
print('\nText fields with embedded commas :', has_comma)
print('Text fields with embedded quotes :', has_quote)

df['text_len'] = df['Text'].str.len()
print('\nText length stats:\n', df['text_len'].describe())
real_newlines = df['Text'].str.contains('\n', na=False).sum()
print('Rows with real newline in text:', real_newlines)
print('Longest comment length:', df['text_len'].max())

df_skip = pd.read_csv('new_ev_tech_cleaned_dataset.csv', on_bad_lines='skip', encoding='utf-8')
df_err = pd.read_csv('new_ev_tech_cleaned_dataset.csv', on_bad_lines='error', encoding='utf-8')
print('\non_bad_lines=error rows:', len(df_err))
print('on_bad_lines=skip rows :', len(df_skip))
print('Malformed rows skipped:', len(df_err) - len(df_skip))

## Step 6: Visualize
Theory: A histogram of text length helps detect short comments and long outliers.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['text_len'].fillna(0), bins=40, color='steelblue', edgecolor='white')
plt.title('Distribution of Comment Length')
plt.xlabel('Text Length (characters)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## Step 7: Interpret
- Delimiter and encoding are correctly handled during load.  
- Key numeric/date columns are cast to usable types.  
- Parent_ID null values are mapped to ROOT for top-level comments.  
- Text is normalized for better NLP readiness.  
- Malformed-row check confirms file reliability.